# AstroML Feature Store — End-to-End Walkthrough

This notebook mirrors the *End-to-End Usage Guide* in [`docs/FEATURE_STORE.md`](../docs/FEATURE_STORE.md) and covers the full lifecycle: **writing features → reading feature sets → caching & purging → online/offline serving → point-in-time training matrix**.

All API calls below use the real `FeatureStore` methods and were validated against `astroml/features/feature_store.py`.

> **Prerequisite:** a working AstroML install (`pip install -e .`). At the time of writing, importing `astroml.features` also imports `astroml.cache`, which currently contains an unrelated syntax error in `graph_cache.py` (issue #767). Fix or stub that to run this notebook end-to-end.

In [ ]:
import tempfile
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

from astroml.features import create_feature_store
from astroml.features.feature_store import FeatureType

np.random.seed(0)
store = create_feature_store(tempfile.mkdtemp() + '/fs')
print('Feature store created at', store.storage.storage_path)

In [ ]:
# ---- Sample transaction data -------------------------------------
n = 100
data = pd.DataFrame({
    'entity_id': [f'a{i % 10}' for i in range(n)],
    'timestamp': [datetime(2024, 1, 1) + timedelta(hours=i) for i in range(n)],
    'amount': np.random.rand(n) * 100,
    'src': [f'a{i % 10}' for i in range(n)],
    'dst': [f'a{(i + 1) % 10}' for i in range(n)],
})
data.head()

## 1. Write features

Register a computer, then `compute_and_store`.

In [ ]:
def balance_computer(df, entity_col, timestamp_col, **kwargs):
    sent = df.groupby('src')['amount'].sum()
    received = df.groupby('dst')['amount'].sum()
    accounts = sorted(set(sent.index) | set(received.index))
    values = [received.get(a, 0.0) - sent.get(a, 0.0) for a in accounts]
    return pd.DataFrame({'account_balance': values}, index=accounts)

store.register_feature(
    name='account_balance',
    computer=balance_computer,
    description='Net inflow minus outflow per account',
    feature_type=FeatureType.NUMERIC,
    tags=['balance', 'financial'],
    owner='notebook',
)

feature_values = store.compute_and_store(
    feature_name='account_balance',
    data=data,
    entity_col='entity_id',
    timestamp_col='timestamp',
)
feature_values.head()

## 2. Read feature sets

In [ ]:
feature_set = store.create_feature_set(
    name='risk_features',
    feature_names=['account_balance'],
    description='Features for risk assessment',
    entity_type='account',
)
print('feature set:', feature_set.name, '-> ids:', feature_set.feature_ids)

entities = data['entity_id'].unique()[:5].tolist()
matrix = store.get_features_for_entities(
    feature_names=['account_balance'], entity_ids=entities, parallel=True
)
matrix

## 3. Caching

The store keeps an in-process LRU+TTL cache. Inspect it with `get_cache_stats()` (there is **no** `store.cache` attribute) and clear it with `clear_cache()`.

In [ ]:
stats = store.get_cache_stats()
print('cached_features:', stats['cached_features'])
print('hit_rate:', round(stats['hit_rate'], 3))

# batch_mode clears the cache before and after the block
with store.batch_mode():
    _ = store.get_feature('account_balance')
print('after batch_mode cached_features:', store.get_cache_stats()['cached_features'])

## 4. Online / offline serving

In [ ]:
# offline Parquet write -> materialize into online store -> online read
# NOTE: write_offline_features defaults to a tz-aware now() if 'timestamp'
# is absent. Add an explicit (tz-naive) compute timestamp so it lines up with
# the naive timestamps we use in get_historical_features below.
offline_df = feature_values.reset_index().rename(columns={'index': 'entity_id'})
offline_df['timestamp'] = pd.Timestamp('2024-01-31')  # tz-naive 'as computed at'
written = store.write_offline_features('account_balance', offline_df)
served = store.materialize_to_online(['account_balance'])
print('offline rows written:', written, '| materialized to online:', served)

online = store.get_online_features(entities, ['account_balance'])
dict(list(online.items())[:2])

## 5. Point-in-time training matrix

`get_historical_features` does an as-of join so each training row only sees feature values computed *at or before* its label timestamp (no leakage). Keep the entity_df timestamps and the offline feature timestamps in the **same timezone awareness**.

In [ ]:
entity_df = pd.DataFrame({
    'entity_id': entities,
    'timestamp': pd.to_datetime(['2024-02-01'] * len(entities)),
})
training_matrix = store.get_historical_features(
    entity_df=entity_df,
    feature_names=['account_balance'],
    entity_col='entity_id',
    timestamp_col='timestamp',
)
training_matrix

## 6. Purging

`FeatureStore` has no single `purge()`; clear the layer you mean (in-process cache, or online-store entries).

In [ ]:
store.clear_cache()
deleted = store.online_store.delete_online_features(
    entities, feature_names=['account_balance']
)
print('online entries deleted:', deleted)
print('cached_features after clear:', store.get_cache_stats()['cached_features'])

## Takeaways

- `compute_and_store` / `register_feature` for the write path.
- `create_feature_set` + `get_features_for_entities` for reads.
- `get_cache_stats` / `clear_cache` / `batch_mode` for the built-in cache.
- `write_offline_features` → `materialize_to_online` → `get_online_features` for serving.
- `get_historical_features` for leak-free training matrices.
- The training pipeline does **not** auto-consume the store yet — call `get_historical_features` yourself to build the feature matrix.